# OpenDrumScribe Stage 2: AI MIDI Assembly Line (Omnizart) 🏭
上传你提取的纯鼓声音频（支持wav, mp3等）。系统将利用 Miniconda 构建绝对纯净的沙盒，自动转码音频，并利用 AI 语义分类输出标准 `.mid` 鼓谱。

**关键步骤：** 运行前请务必点击顶部菜单栏 `代码执行程序` > `更改附加到运行时的类型` > 选择 `T4 GPU`。

In [ ]:
# 1. Initialize AI Model via Miniconda Sandbox (构建绝对纯净的虚拟机隔离环境)
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local/miniconda > /dev/null 2>&1
!sudo apt-get install libsndfile-dev fluidsynth ffmpeg -y > /dev/null 2>&1

# 创建名为 omni_env 的纯净 Python 3.8 环境
!/usr/local/miniconda/bin/conda create -n omni_env python=3.8 -y > /dev/null 2>&1

# 在虚拟环境中安装依赖（保留日志输出）
!/usr/local/miniconda/envs/omni_env/bin/pip install numpy Cython
!/usr/local/miniconda/envs/omni_env/bin/pip install git+https://github.com/Music-and-Culture-Technology-Lab/omnizart.git

# 下载模型权重
!/usr/local/miniconda/envs/omni_env/bin/omnizart download-checkpoints
print("\n--- AI Model Ready via Miniconda! / Miniconda 纯净沙盒构建完毕，模型就绪！ ---")

In [ ]:
# 2. Upload Audio & Auto-Normalize (上传任意音频格式，系统底层自动转码为标准WAV)
from google.colab import files
import os
import subprocess

if os.path.exists('drums_input.wav'):
    os.remove('drums_input.wav')

uploaded = files.upload()
original_file = next(iter(uploaded))
print(f"\nProcessing {original_file}...")

# 强制重采样并转码为AI最易识别的格式 (44.1kHz, 单声道, WAV)
subprocess.run(['ffmpeg', '-i', original_file, '-ar', '44100', '-ac', '1', 'drums_input.wav', '-y'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("\n--- Audio normalized to WAV and ready for transcription! / 音频已自动转码为标准WAV格式，准备执行转谱！ ---")

In [ ]:
# 3. Run Semantic Classification (调用 Miniconda 环境内的绝对路径执行器)
!/usr/local/miniconda/envs/omni_env/bin/omnizart drum transcribe drums_input.wav

In [ ]:
# 4. Download Standard MIDI (下载完美映射的鼓谱 MIDI)
from google.colab import files
import os

if os.path.exists('drums_input.mid'):
    print("Success! Downloading MIDI... / 转换成功！正在下载标准 MIDI 文件...")
    files.download('drums_input.mid')
else:
    print("Error: MIDI not generated. Please check the logs. / 错误：未能生成 MIDI，请检查上方报错信息。")